# H&E Tumor Invasion Front Detection – Training & Evaluation

This notebook orchestrates the deep-learning pipeline using the reusable modules
in `multiplex_pipeline.hne`:

1. **Data loading** – scan the balanced patch directory and split into train / test.
2. **Model** – instantiate the U-Net with SE-attention blocks.
3. **Training** – mixed-precision training with gradient accumulation.
4. **Inference** – batch prediction with morphological post-processing.
5. **Metrics** – IoU, Dice, confusion matrix, ROC/AUC, per-class classification report.
6. **Overlays** – side-by-side original / predicted / ground-truth visualisations.


In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import torch
from sklearn.model_selection import train_test_split

from multiplex_pipeline.hne.models.unet import UNet
from multiplex_pipeline.hne.data.dataset import PatchDataset, get_valid_pairs
from multiplex_pipeline.hne.training.trainer import train_model
from multiplex_pipeline.hne.inference.predictor import predict_patches
from multiplex_pipeline.hne.analysis.metrics import (
    compute_confusion_matrix,
    compute_iou_dice,
    compute_classification_report,
    compute_overall_metrics,
    compute_roc_auc,
    compute_per_patch_iou,
)
from multiplex_pipeline.hne.visualization.overlays import save_batch_overlays
from multiplex_pipeline.hne.config import (
    DEFAULT_EPOCHS, LEARNING_RATE, BATCH_SIZE, GRAD_ACCUMULATION,
    RANDOM_SEED, MODELS_DIR, OVERLAYS_DIR, METRICS_DIR,
)

## 1 – Configuration

In [ ]:
PATCH_DIR  = Path(r"C:\Temp\patches_2048\HC22B0006126_balanced")
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS     = DEFAULT_EPOCHS

# Set to a .pth path to skip training and load an existing model:
LOAD_MODEL_PATH = None

for d in (MODELS_DIR, OVERLAYS_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")

## 2 – Data loading & train/test split

In [ ]:
pairs = get_valid_pairs(PATCH_DIR)
print(f"Valid patch quads: {len(pairs)}")

# Stratify by dominant class
strat = []
for *_, msk in pairs:
    gt = cv2.imread(msk, 0)
    gt = np.where(np.isin(gt, [0, 1, 2]), gt, -1)
    u, c = np.unique(gt, return_counts=True)
    valid = [(u_, c_) for u_, c_ in zip(u, c) if u_ in [0, 1, 2]]
    strat.append(max(valid, key=lambda x: x[1])[0] if valid else 0)

train_pairs, test_pairs = train_test_split(
    pairs, test_size=0.2, stratify=strat, random_state=RANDOM_SEED
)
print(f"Train: {len(train_pairs)}  |  Test: {len(test_pairs)}")

ds_train = PatchDataset(train_pairs, augment=True)
ds_test  = PatchDataset(test_pairs,  augment=False)

## 3 – Model initialisation

In [ ]:
model = UNet().to(DEVICE)
print(model)

## 4 – Training  (or load existing checkpoint)

In [ ]:
if LOAD_MODEL_PATH is not None:
    from multiplex_pipeline.hne.io.loaders import load_model
    model = load_model(LOAD_MODEL_PATH, device=DEVICE)
    print(f"Loaded model from {LOAD_MODEL_PATH}")
else:
    history = train_model(
        model, ds_train, ds_test,
        device=DEVICE,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        grad_accumulation=GRAD_ACCUMULATION,
    )
    checkpoint = MODELS_DIR / "unet_trained.pth"
    torch.save(model.state_dict(), checkpoint)
    print(f"Model saved to {checkpoint}")

## 5 – Inference on test set

In [ ]:
y_true, y_pred, y_prob = predict_patches(
    model, ds_test,
    device=DEVICE,
    output_dir=OVERLAYS_DIR,
    apply_ma=True,
)
print(f"Predicted {y_true.shape[0]:,} pixels.")

## 6 – Evaluation metrics

In [ ]:
cm = compute_confusion_matrix(y_true, y_pred)
print("Confusion matrix:")
print(cm)

iou_dice = compute_iou_dice(y_true, y_pred)
print("\nIoU per class:",  iou_dice["iou"])
print("Dice per class:", iou_dice["dice"])

report = compute_classification_report(y_true, y_pred)
import pandas as pd
print("\nPer-class metrics:")
print(pd.DataFrame(report).T)

overall = compute_overall_metrics(y_true, y_pred)
print("\nOverall metrics:", overall)

auc_scores = compute_roc_auc(y_true, y_prob)
print("\nROC AUC per class:", auc_scores)

## 7 – Per-patch IoU

In [ ]:
n_patches  = len(test_pairs)
patch_size = y_true.shape[0] // n_patches
t_patches  = [y_true[i*patch_size:(i+1)*patch_size] for i in range(n_patches)]
p_patches  = [y_pred[i*patch_size:(i+1)*patch_size] for i in range(n_patches)]

patch_ious = compute_per_patch_iou(t_patches, p_patches)
print("Mean IoU – Invasion Front:", patch_ious[:, 1].mean().round(3))
print("Mean IoU – Stroma:",         patch_ious[:, 2].mean().round(3))

## 8 – Prediction overlays

In [ ]:
# Reconstruct per-patch 2D arrays for overlay generation
import math
side = int(math.sqrt(patch_size))

rgb_paths   = [p[0] for p in test_pairs]
pred_masks  = [y_pred[i*patch_size:(i+1)*patch_size].reshape(side, side) for i in range(n_patches)]
gt_masks    = [y_true[i*patch_size:(i+1)*patch_size].reshape(side, side) for i in range(n_patches)]

save_batch_overlays(rgb_paths, pred_masks, gt_masks, output_dir=OVERLAYS_DIR)
print(f"Overlays saved to {OVERLAYS_DIR}")